# 09 - The ML Trap Table

Learn from common ML mistakes and how to avoid them.

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


## The traps

| Trap | Symptom | Prevention |
|------|---------|------------|
| Data leakage | High accuracy but fails in production | Use time-based splits |
| Overfitting | Train accuracy high, test low | Use cross-validation |
| Baseline ignored | Model beats random but not baseline | Always compare to baselines |
| Accuracy obsession | High accuracy but poor recall | Use precision, recall, F1 |
| Feature engineering | Features don't help | Check feature importance |
| Data drift | Model works offline, fails online | Monitor performance over time |
| Correlation != causation | Spurious relationships | Understand data generation |
| Dimensionality curse | Too many features, not enough data | Use feature selection/PCA |

In [2]:
# Let's demonstrate some traps!

print("=" * 60)
print("ML Trap Demonstrations")
print("=" * 60)

# Trap 1: Random split vs time-based split
print("\nTrap 1: Random split overstates accuracy")
print("-" * 50)

# Load data
conn = psycopg.connect(dsn())
query = "SELECT observed_at, site_id,"
query += " band_25hz, band_50hz, band_160hz"
query += " FROM acoustic_tol_hourly WHERE site_id = 'mb01' ORDER BY observed_at"

acoustic_df = pd.read_sql_query(query, conn)

query2 = "SELECT observed_at, presence FROM detection_hourly"
query2 += " WHERE taxon = 'dolphin' AND is_event_list = FALSE ORDER BY observed_at"

dolphin_df = pd.read_sql_query(query2, conn)
conn.close()

merged = acoustic_df.merge(dolphin_df, on='observed_at')
X = merged[[c for c in merged.columns if c.startswith('band_')]].values
y = merged['presence'].values

# Time-based split
split_idx = int(len(X) * 0.75)
X_train_t, X_test_t, y_train_t, y_test_t = X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

# Random split
from sklearn.model_selection import train_test_split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)

# Train and evaluate
def evaluate_model(name, X_tr, X_te, y_tr, y_te):
    model = LogisticRegression(max_iter=2000)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return accuracy_score(y_te, y_pred)

acc_time = evaluate_model("Time-based", X_train_t, X_test_t, y_train_t, y_test_t)
acc_rand = evaluate_model("Random", X_train_r, X_test_r, y_train_r, y_test_r)

print(f"  Time-based split accuracy:  {acc_time:.3f}")
print(f"  Random split accuracy:      {acc_rand:.3f}")
print(f"  Overstatement:              {acc_rand - acc_time:+.3f}")

if acc_rand > acc_time:
    print("  -> Random split overstates accuracy (common problem!)")
else:
    print("  -> In this case, random split is worse (less variance in test)")

# Trap 2: Always predict majority
print("\n" + "=" * 60)
print("Trap 2: Ignoring the majority baseline")
print("-" * 50)

y_majority = np.zeros_like(y_test_t)
majority_acc = accuracy_score(y_test_t, y_majority)

print(f"  Test positive rate:         {y_test_t.mean():.2%}")
print(f"  Majority baseline:          {majority_acc:.3f}")
print(f"  Model accuracy:             {acc_time:.3f}")

if acc_time - majority_acc < 0.05:
    print("  -> Model shows little improvement over baseline!")
    print("  -> Is the model learning anything real?")
else:
    print("  -> Model significantly beats baseline")

# Trap 3: Feature importance without context
print("\n" + "=" * 60)
print("Trap 3: One feature does all the work")
print("-" * 50)

# Check if one band dominates
band_importance = []
for col in [c for c in merged.columns if c.startswith('band_')]:
    X_single = merged[[col]].values
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_single, y, test_size=0.25, random_state=42)
    model_s = LogisticRegression(max_iter=2000)
    model_s.fit(X_train_s, y_train_s)
    pred_s = model_s.predict(X_test_s)
    band_importance.append((col, accuracy_score(y_test_s, pred_s)))

band_importance.sort(key=lambda x: x[1], reverse=True)
best_band, best_acc = band_importance[0]

print(f"  Best single band: {best_band}")
print(f"  Single band accuracy: {best_acc:.3f}")
print(f"  Multi-band accuracy:  {acc_rand:.3f}")

if best_acc > acc_rand * 0.95:
    print("  -> Model relies on one band (the 20 kHz trap!)")
    print("  -> Check if band overlaps with target signal")
else:
    print("  -> Model uses multiple features")

ML Trap Demonstrations

Trap 1: Random split overstates accuracy
--------------------------------------------------
  Time-based split accuracy:  0.800
  Random split accuracy:      0.778
  Overstatement:              -0.022
  -> In this case, random split is worse (less variance in test)

Trap 2: Ignoring the majority baseline
--------------------------------------------------
  Test positive rate:         19.74%
  Majority baseline:          0.803
  Model accuracy:             0.800
  -> Model shows little improvement over baseline!
  -> Is the model learning anything real?

Trap 3: One feature does all the work
--------------------------------------------------
  Best single band: band_50hz
  Single band accuracy: 0.780
  Multi-band accuracy:  0.778
  -> Model relies on one band (the 20 kHz trap!)
  -> Check if band overlaps with target signal


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_26308/789404616.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  acoustic_df = pd.read_sql_query(query, conn)
/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_26308/789404616.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dolphin_df = pd.read_sql_query(query2, conn)


## The habits that prevent traps

1. Time-based splits - Don't shuffle time-series data
2. Compare to baselines - Can a simple rule beat your model?
3. Check feature importance - Does one feature do all the work?
4. Use proper metrics - Precision, recall, F1 for imbalanced data
5. Cross-validate - Check stability across different splits
6. Understand data generation - Correlation != causation

## Your turn

Build a model and check for these traps!